# Data Preparation on a Housing Dataset

You'll work with a synthetic dataset about houses and prepare it for modeling, one step at a time: **missing values → encoding → scaling → a pipeline that does it all**. (You explored data like this in week 4, so we skip straight to preparation.)

The dataset contains the following columns:
- 'area': House area in square feet (numeric)
- 'bedrooms': Number of bedrooms (numeric)
- 'age': Age of the house in years (numeric)
- 'neighborhood': Categorical feature representing different neighborhoods
- 'distance_to_city_center': Distance to city center in miles (numeric)
- 'price': House price in thousands of dollars (target variable)

Some values are missing on purpose. Run the next cell to create the data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

# Generate synthetic data
np.random.seed(42)
n_samples = 1000

area = np.random.uniform(1000, 5000, n_samples)
bedrooms = np.random.randint(1, 6, n_samples)
age = np.random.uniform(1, 50, n_samples)
neighborhood = np.random.choice(['A', 'B', 'C', 'D', 'E'], n_samples)
distance_to_city_center = np.random.exponential(scale=5, size=n_samples)

price = (
    10 * np.log(area) +
    5 * bedrooms -
    2 * age +
    np.where(neighborhood == 'A', 50, 0) +
    np.where(neighborhood == 'C', 30, 0) +
    np.where(neighborhood == 'E', 10, 0) +
    np.where(neighborhood == 'D', -20, 0) +
    np.where(neighborhood == 'B', -40, 0) -
    20 * np.log(distance_to_city_center + 1) +
    np.random.normal(0, 10, n_samples)
)

# Create DataFrame
df = pd.DataFrame({
    'area': area,
    'bedrooms': bedrooms,
    'age': age,
    'neighborhood': neighborhood,
    'distance_to_city_center': distance_to_city_center,
    'price': price
})

# Add some missing values

df.loc[np.random.choice(df.index, 50, replace=False), 'neighborhood'] = np.nan
df.loc[np.random.choice(df.index, 30, replace=False), 'bedrooms'] = np.nan
df.loc[np.random.choice(df.index, 20, replace=False), 'age'] = np.nan

## **Stage 1: Handling Missing Values**

In the following, you'll identify missing values, visualize them, and apply basic strategies to handle them.

Your tasks:
1. Identify missing values in the dataset
2. Visualize the extent of missing data
3. Apply simple strategies to handle missing values
4. Verify that all missing values have been addressed

#### Exercise 1: Check for missing values in each column

- Hint: Use df.isnull().sum()

In [ ]:
# TODO: Check for missing values in each column
# Hint: Use df.isnull().sum()
missing_values = # Your code here
print("Missing values in each column:")
print(missing_values)

#### Exercise 2: Calculate the percentage of missing values in each column

- Hint: Divide the missing value counts by the total number of rows and multiply by 100

In [ ]:
missing_percentages = # Your code here
print("\nPercentage of missing values in each column:")
print(missing_percentages)

# Visualize missing values
plt.figure(figsize=(10, 6))
missing_percentages.plot(kind='bar')
plt.title('Percentage of Missing Values by Column')
plt.xlabel('Columns')
plt.ylabel('Percentage of Missing Values')
plt.show()

#### Exercise 3: Handle missing values

Fill the missing values in a copy of the data, `df_imputed`: the most frequent value for `neighborhood`, the median for `bedrooms` and `age`.

In [ ]:
# Handle missing values using SimpleImputer.
# We work on a COPY (df_imputed) so the original df keeps its missing values for Exercise 6.
df_imputed = df.copy()

# For categorical data (neighborhood)


# TODO: Create a SimpleImputer with strategy='most_frequent' for categorical data
cat_imputer = # Your code here

# Note the double square brackets on the left!
df_imputed[['neighborhood']] = cat_imputer.fit_transform(df[['neighborhood']])

# For numeric data (bedrooms and age)
numeric_columns = ['bedrooms', 'age']


# TODO: Create a SimpleImputer with strategy='median' for numeric data
num_imputer = # Your code here
df_imputed[numeric_columns] = num_imputer.fit_transform(df_imputed[numeric_columns])

# Verify that all missing values have been handled
print("\nMissing values after imputation:")
print(df_imputed.isnull().sum())

# TODO: Create a function to check if a dataframe has any missing values
def has_missing_values(dataframe):
    # Your code here
    pass

# Test the function
print("\nDoes the dataframe have any missing values?", has_missing_values(df_imputed))

# Bonus: Compare imputed values with original distribution
plt.figure(figsize=(12, 4))

plt.subplot(131)
sns.histplot(data=df_imputed, x='neighborhood', kde=True)
plt.title('Imputed Neighborhood Distribution')

plt.subplot(132)
sns.histplot(data=df_imputed, x='bedrooms', kde=True)
plt.title('Imputed Bedrooms Distribution')

plt.subplot(133)
sns.histplot(data=df_imputed, x='age', kde=True)
plt.title('Imputed Age Distribution')

plt.tight_layout()
plt.show()

## **Stage 2: Encoding Data**

#### Exercise 4: Use a OneHotEncoder to transform categorical columns
In this exercise, you'll use scikit-learn's OneHotEncoder to encode categorical variables. We'll focus on the 'neighborhood' column from our housing dataset.
One-hot encoding creates binary columns for each category in a categorical variable. This is particularly useful when there's no ordinal relationship between the categories.
Your tasks:

- Apply OneHotEncoder to the 'neighborhood' column
- Examine the encoding and its impact on the data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
import matplotlib.pyplot as plt
import seaborn as sns

# Uses df_imputed from Exercise 3


print(df_imputed['neighborhood'].unique())

# TODO: Create an instance of OneHotEncoder
# Hint: OneHotEncoder(sparse_output=False)
onehot_encoder = # Your code here

# TODO: Fit the encoder on the 'neighborhood' column and transform it
# Hint: Use fit_transform() method - be sure to only transform the "neighborhood" values
encoded_neighborhood = # Your code here

# Get the feature names from the encoder
feature_names = # Your code here

# Create a new dataframe with the encoded features
encoded_df = pd.DataFrame(encoded_neighborhood, columns=feature_names, index=df_imputed.index)

# Concatenate the encoded features with the original dataframe
df_encoded = pd.concat([df_imputed, encoded_df], axis=1)

# Display the first few rows to see the original and encoded values
print("\nFirst few rows with original and encoded 'neighborhood':")
print(df_encoded[['neighborhood'] + list(feature_names)].head(10))

# Print the encoding mapping
print("\nEncoding mapping:")
for category, encoded_cols in zip(onehot_encoder.categories_[0], encoded_df.columns):
    print(f"{category}: {encoded_cols}")

## **Stage 3: Scaling Data**

### Exercise 5: Feature Scaling with StandardScaler
In this exercise, you'll use scikit-learn's StandardScaler to normalize numerical features. Scaling is an important preprocessing step for many machine learning algorithms, especially when features are on different scales.
Your tasks:

- Identify which features need scaling
- Apply StandardScaler to the appropriate features
- Visualize the effect of scaling on the distribution of the features
- Compare the correlation of scaled features with the target variable. (Look closely at the result: why is it what it is?)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

numerical_features = ['area', 'bedrooms', 'age', 'distance_to_city_center']

# TODO: Create an instance of StandardScaler
scaler = # Your code here

# TODO: Fit the scaler on the numerical features and transform them
# Hint: Use fit_transform() method
scaled_features = # Your code here

# Create a new dataframe with the scaled features
df_scaled = pd.DataFrame(scaled_features, columns=numerical_features, index=df_imputed.index)

# Add the target variable and categorical features to the scaled dataframe
df_scaled['price'] = df_imputed['price']
df_scaled['neighborhood'] = df_imputed['neighborhood']

# Display the first few rows of the original and scaled dataframes
print("Original data:")
print(df_imputed[numerical_features + ['price']].head())
print("\nScaled data:")
print(df_scaled[numerical_features + ['price']].head())

# Visualize the effect of scaling
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Distribution of Features Before and After Scaling')

for i, feature in enumerate(numerical_features):
    sns.histplot(df_imputed[feature], kde=True, ax=axes[i//2, i%2], color='blue', alpha=0.5, label='Original')
    sns.histplot(df_scaled[feature], kde=True, ax=axes[i//2, i%2], color='red', alpha=0.5, label='Scaled')
    axes[i//2, i%2].set_title(feature)
    axes[i//2, i%2].legend()

plt.tight_layout()
plt.show()

# TODO: Compare correlation of original and scaled features with the target variable
# Hint: Use df_imputed[numerical_features + ['price']].corr(numeric_only=True)['price'] for original
# and df_scaled[numerical_features + ['price']].corr(numeric_only=True)['price'] for scaled
original_correlation = # Your code here
scaled_correlation = # Your code here

print("\nCorrelation with price (original vs scaled):")
print(pd.DataFrame({'Original': original_correlation, 'Scaled': scaled_correlation}))

# Bonus: Visualize the correlation comparison
plt.figure(figsize=(10, 6))
bar_width = 0.35
index = np.arange(len(numerical_features))

plt.bar(index, original_correlation[numerical_features], bar_width, label='Original', alpha=0.8)
plt.bar(index + bar_width, scaled_correlation[numerical_features], bar_width, label='Scaled', alpha=0.8)

plt.xlabel('Features')
plt.ylabel('Correlation with Price')
plt.title('Correlation Comparison: Original vs Scaled Features')
plt.xticks(index + bar_width/2, numerical_features, rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## **Stage 4: Modeling data**

### Exercise 6: Building a Basic sklearn Pipeline

The preceding steps - imputation, encoding, and scaling, are commonly applied in ML tasks. Because this can become cumbersome, to manage all of the different data artifacts, sklearn provides a `Pipeline` class that allows us to bundle all of these steps together along with an ML algorithm.  The `Pipeline` follows the estimator API, and can therefore be used just like any ML algorithm. `Pipeline`'s also help to avoid what is known as the 'data leakage' problem, where test data contaminates the training data.  We'll talk a bit more about data leakage later.

In this exercise, you'll create a simple sklearn Pipeline that combines preprocessing steps with a machine learning model. 

Your tasks:

- Create a pipeline that includes imputation, OneHotEncoder, StandardScaler, and a simple model (e.g., LinearRegression), starting from the original data *with* its missing values
- Fit the pipeline on the training data
- Make predictions using the pipeline
- Evaluate the model's performance

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Note: we start again from the ORIGINAL df, missing values and all. The pipeline handles them.


# Split the data into features (X) and target (y)
X = df.drop('price', axis=1)
y = df['price']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the feature types
numeric_features = ['area', 'bedrooms', 'age', 'distance_to_city_center']
categorical_features = ['neighborhood']

# TODO: Create a ColumnTransformer for preprocessing

# First, we'll create a pipeline to do imputation / scaling on the numeric features.
# Hint: Use make_pipeline(SimpleImputer(strategy="median"),StandardScaler())

num_pipeline = # Your code here

# Second, create a pipeline for the categorical features.  Use SimpleImputer with
# the 'most frequent' strategy followed by a OneHotEncoder

cat_pipeline = # Your code here

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numeric_features),
        ('cat', cat_pipeline, categorical_features)
    ])

# TODO: Create a pipeline that includes the preprocessor and a LinearRegression model
# Hint: Use Pipeline([('preprocessor', preprocessor), ('regressor', LinearRegression())])
pipeline = # Your code here

# TODO: Fit the pipeline on the training data
# Hint: Use the 'fit' method
# Your code here

# TODO: Make predictions on the test data
# Hint use the 'predict' method
y_pred = # Your code here

# TODO: Evaluate the model's performance
# Hint see sklearn's metrics
# See https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html
# See https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html


mse = # Your code here
r2 = # Your code here

print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared Score: {r2:.2f}")

# Bonus: Feature importance analysis
# Get feature names after preprocessing
feature_names = preprocessor.get_feature_names_out()

# Get coefficients from the linear regression model
coefficients = pipeline.named_steps['regressor'].coef_

# Create a dataframe of feature importances
feature_importance = pd.DataFrame({'feature': feature_names, 'importance': abs(coefficients)})
feature_importance = feature_importance.sort_values('importance', ascending=False)

# Visualize feature importances
plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=feature_importance.head(10))
plt.title('Top 10 Feature Importances')
plt.tight_layout()
plt.show()